# 70. 時系列Gaussian Processと時間カーネル

不規則な観測時刻を持つデータを使い、時間カーネル、学習、将来予測、固定時刻での候補提案を確認します。時間は制御変数とは区別します。理論は [カーネル](../../docs/theory/03_kernel.md)、実装構造は [Guide](../../docs/guide/03_kernel.md) を参照してください。

In [ ]:
import torch
import matplotlib.pyplot as plt
from botorch.acquisition.analytic import LogExpectedImprovement
from botorch.acquisition.logei import qLogExpectedImprovement
from botorch.optim import optimize_acqf
from botorch.sampling.normal import SobolQMCNormalSampler
from gpytorch.kernels import RBFKernel, ScaleKernel
from robotorchan.models.standard.single_task import SingleTaskGP, MixedSingleTaskGP
from robotorchan.temporal import make_temporal_kernel, TimeAxis, elapsed_lag_features

torch.manual_seed(70)
torch.set_num_threads(1)
dtype = torch.double
t = torch.sort(torch.rand(24, dtype=dtype) * 2.5).values
def truth(t):
    return 0.35 * t + torch.sin(2 * torch.pi * t)
y = truth(t) + 0.07 * torch.randn_like(t)
grid = torch.linspace(0, 3.5, 180, dtype=dtype)
plt.scatter(t, y, label="observed")
plt.plot(grid, truth(grid), label="truth")
plt.axvline(t.max(), linestyle="--", label="last observation")
plt.legend()
plt.show()

## 1. 時間軸と不規則ラグ

時間座標の原点・単位を決める処理と、ラグ特徴量を作る処理は別です。以下は学習区間だけで正規化し、観測が存在しない時刻の特徴量を作る例です。

In [ ]:
axis = TimeAxis.fit(t)
print("scaled future time:", axis.transform(torch.tensor([3.0], dtype=dtype)))
features, valid = elapsed_lag_features(
    t, y[:, None], torch.tensor([2.6, 2.8], dtype=dtype), lags=[0.0, 0.5]
)
print(features, valid)

## 2. 共分散の違い

時間差に対する共分散を8種類比較します。線形カーネルは定常ではないため、固定した参照時刻との共分散を描きます。Spectral Mixtureの初期値は学習前の例であり、周期の推定結果ではありません。

In [ ]:
kinds = ["rbf", "matern", "exponential", "rq", "periodic",
         "locally_periodic", "linear", "spectral_mixture"]
delta = torch.linspace(0, 3, 150, dtype=dtype)[:, None]
fig, axes = plt.subplots(4, 2, figsize=(11, 10))
for kind, ax in zip(kinds, axes.flat):
    kwargs = {"period_length": 1.0} if kind in ("periodic", "locally_periodic") else {}
    kernel = make_temporal_kernel(kind, **kwargs).double()
    with torch.no_grad():
        cov = kernel(delta + 1, torch.ones(1, 1, dtype=dtype)).to_dense().squeeze(-1)
    ax.plot(delta[:, 0], cov)
    ax.set_title(kind)
    ax.set_xlabel("time lag")
fig.tight_layout()
plt.show()

## 3. 不規則時系列の予測

RBF、周期、局所周期、トレンドと周期の和を比較します。信頼帯は潜在関数の周辺事後標準偏差の1.96倍であり、同時信頼帯や観測ノイズ込みの予測区間ではありません。最終観測時刻以降の外挿に注目してください。

In [ ]:
specs = {
    "RBF": make_temporal_kernel("rbf"),
    "Periodic": make_temporal_kernel("periodic", period_length=1.0),
    "Locally periodic": make_temporal_kernel("locally_periodic", period_length=1.0),
    "Trend + seasonal": (
        make_temporal_kernel("linear")
        + make_temporal_kernel("locally_periodic", period_length=1.0)
    ),
}
fig, axes = plt.subplots(2, 2, figsize=(12, 7))
models = {}
for label, kernel in specs.items():
    model = SingleTaskGP(t[:, None], y[:, None], covar_module=kernel.double())
    model.fit(optimizer_kwargs={"options": {"maxiter": 40}})
    models[label] = model
for (label, model), ax in zip(models.items(), axes.flat):
    with torch.no_grad():
        posterior = model.posterior(grid[:, None])
        mean = posterior.mean.squeeze(-1)
        sd = posterior.variance.sqrt().squeeze(-1)
    ax.plot(grid, truth(grid), "--", label="truth")
    ax.scatter(t, y, s=15, label="observed")
    ax.plot(grid, mean, label="mean")
    ax.fill_between(grid, mean - 1.96 * sd, mean + 1.96 * sd, alpha=0.2)
    ax.axvline(t.max(), linestyle=":")
    ax.set_title(label)
    ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

## 4. 時系列を含む単発BO

時刻を固定し、制御変数だけを最適化します。時刻と制御変数のカーネルの積は交互作用を表現できます。過去の全観測の最大値を改善基準とする例で、時間変動する基準や動的方策とは異なります。

In [ ]:
def objective(x):
    time, u = x[..., 0], x[..., 1]
    return (torch.sin(2 * torch.pi * time) - (u - 0.7)**2).unsqueeze(-1)

u = torch.rand(len(t), dtype=dtype)
X = torch.stack((t, u), dim=-1)
Y = objective(X)
kernel = (
    make_temporal_kernel("locally_periodic", period_length=1.0)
    + ScaleKernel(RBFKernel(active_dims=(1,)))
).double()
model = SingleTaskGP(X, Y, covar_module=kernel)
model.fit(optimizer_kwargs={"options": {"maxiter": 40}})
bounds = torch.tensor([[2.5, 0.0], [3.0, 1.0]], dtype=dtype)
acq = LogExpectedImprovement(model, best_f=Y.max())
candidate, score = optimize_acqf(
    acq, bounds, q=1, num_restarts=3, raw_samples=32, fixed_features={0: 2.75}
)
new_y = objective(candidate)
print("candidate:", candidate, "observed value:", new_y)
line = torch.stack((torch.full_like(grid, 2.75), grid / 3.5), dim=-1)
with torch.no_grad():
    p = model.posterior(line)
    m, s = p.mean[:, 0], p.variance[:, 0].sqrt()
plt.plot(line[:, 1], objective(line)[:, 0], "--", label="truth")
plt.plot(line[:, 1], m, label="mean")
plt.fill_between(line[:, 1], m - 1.96 * s, m + 1.96 * s, alpha=0.2)
plt.scatter(candidate[:, 1].detach(), new_y[:, 0].detach(), label="candidate")
plt.xlabel("control")
plt.legend()
plt.show()

## 5. Mixed変数と同時バッチ提案

カテゴリごとに固定してq=2の同時候補を求め、獲得関数値で比較します。同一バッチ内でカテゴリを変える探索ではない点に注意してください。

In [ ]:
cat = torch.arange(len(t), dtype=dtype) % 2
mixed_X = torch.cat((X, cat[:, None]), dim=-1)
def mixed_objective(x):
    return objective(x) + 0.25 * x[..., 2:3]
mixed_Y = mixed_objective(mixed_X)
def continuous_kernel(batch_shape, num_dims, active_dims):
    return (
        make_temporal_kernel("periodic", batch_shape=batch_shape, time_dim=0, period_length=1.0)
        * make_temporal_kernel("rbf", batch_shape=batch_shape, time_dim=1)
    )
mixed = MixedSingleTaskGP(mixed_X, mixed_Y, cat_dims=[2], cont_kernel_factory=continuous_kernel)
mixed.fit(optimizer_kwargs={"options": {"maxiter": 30}})
mixed_acq = qLogExpectedImprovement(
    mixed, best_f=mixed_Y.max(), sampler=SobolQMCNormalSampler(torch.Size([32]), seed=70)
)
mixed_bounds = torch.tensor([[2.5, 0.0, 0.0], [3.0, 1.0, 1.0]], dtype=dtype)
choices = [
    optimize_acqf(mixed_acq, mixed_bounds, q=2, num_restarts=2, raw_samples=16,
                  fixed_features={0: 2.75, 2: category}, options={"maxiter": 30})
    for category in (0.0, 1.0)
]
candidate, score = max(choices, key=lambda item: item[1].item())
new_y = mixed_objective(candidate)
print(candidate, new_y)
assert candidate.shape == (2, 3)

## 6. 独立多出力GP

同じ時刻で二つの応答を観測する例です。独立出力のバッチ共分散は出力間の相関を学習しません。出力間相関を扱う場合は多タスクGPが必要です。

In [ ]:
Y2 = torch.stack((y, torch.cos(2 * torch.pi * t)), dim=-1)
kernel2 = make_temporal_kernel("matern", batch_shape=torch.Size([2])).double()
multi = SingleTaskGP(t[:, None], Y2, covar_module=kernel2)
multi.fit(optimizer_kwargs={"options": {"maxiter": 30}})
with torch.no_grad():
    p = multi.posterior(grid[:, None])
    mu, sd = p.mean, p.variance.sqrt()
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for j, ax in enumerate(axes):
    ax.scatter(t, Y2[:, j], label="observed")
    ax.plot(grid, mu[:, j], label="mean")
    ax.fill_between(grid, mu[:, j] - 1.96 * sd[:, j],
                    mu[:, j] + 1.96 * sd[:, j], alpha=0.2)
    ax.axvline(t.max(), linestyle=":")
    ax.set_title(f"Output {j}")
    ax.legend()
plt.show()

## 7. まとめ

時間の単位、観測時刻と意思決定時刻の区別、外挿時の不確実性、既知周期の妥当性を確認してください。単一の合成例でカーネルの優劣を結論づけることはできません。